<a href="https://colab.research.google.com/github/hj090/DACON_predict_14class/blob/dev_hj/stage1%2Bstage2%2B%EC%B6%94%EB%A1%A0(07_11).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 셀 A — numpy부터 먼저 정확히 고정 (이게 젤 먼저)
!pip install -q "numpy==1.26.4"

In [ ]:
# 셀 B — 나머지
!pip install -q "scipy==1.15.3" "threadpoolctl==3.6.0" "joblib==1.5.3" "scikit-learn==1.8.0"
!pip install -q "transformers==4.46.3"
!pip uninstall -y torchvision -q

In [ ]:
# 셀 C — ONNX 관련 (원래 #0)
!pip install onnxmltools skl2onnx onnxruntime onnxconverter-common --break-system-packages -q

In [ ]:
# 셀 C 다음, 라이브러리 불러오기 전에 삽입
import numpy
print("numpy:", numpy.__version__)   # 1.26.4로 나와야 함

numpy: 1.26.4


In [ ]:
# 확인 셀(numpy: 2.0.2로 나온 셀) 다음에 새 셀 추가
!pip install -q "numpy==1.26.4"

import numpy
print("numpy:", numpy.__version__)   # 다시 1.26.4로 나와야 함

numpy: 1.26.4


# Stage1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##1. 라이브러리 불러오기

In [ ]:
import csv
import json
import os

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
import numpy as np
import re
import pandas as pd


# 예측 대상 14개 클래스 (Macro-F1 계산에 사용)
ALL_CLASSES = [
    "read_file", "grep_search", "list_directory", "glob_pattern",
    "edit_file", "write_file", "apply_patch",
    "run_bash", "run_tests", "lint_or_typecheck",
    "ask_user", "plan_task", "web_search", "respond_only",
]

##2. 데이터 불러오기



In [ ]:
DATA_DIR = "/content/drive/MyDrive/nlp_competition/data" ### 임유미 수정

# train.jsonl: 한 줄 = 샘플 하나
samples = [json.loads(line)
           for line in open(os.path.join(DATA_DIR, "train.jsonl"), encoding="utf-8")
           if line.strip()]

# train_labels.csv: id -> action 매핑
labels = {row["id"]: row["action"]
          for row in csv.DictReader(open(os.path.join(DATA_DIR, "train_labels.csv"), encoding="utf-8"))}

# 입력 X = current_prompt, 정답 y = action
X = [s["current_prompt"] for s in samples]
y = [labels[s["id"]] for s in samples]
ids = [s["id"] for s in samples]

# id로 원본 샘플(session_meta 포함)을 바로 찾기 위한 딕셔너리
samples_by_id = {s["id"]: s for s in samples}


print("samples:", len(X), "| classes:", len(set(y)))

samples: 70000 | classes: 14


##3. 학습 / 검증 데이터 분할

In [ ]:
X_train, X_val, y_train, y_val, train_ids, val_ids = train_test_split(
    X, y, ids, test_size=0.2, stratify=y, random_state=42,
)
print("train:", len(X_train), "| val:", len(X_val))

train: 56000 | val: 14000


In [ ]:
# session meta 추가 연결
def extract_session_features(sample):
  s = samples_by_id[sample]

  meta = s.get("session_meta", {}) or {}
  ws = meta.get("workspace", {}) or {}
  lang_mix = ws.get("language_mix") or {}
  dom_lang, dom_ratio = max(lang_mix.items(), key=lambda kv: kv[1]) if lang_mix else ("none", 0.0)

  return {
    "language_pref": meta.get("language_pref", "unknown"),
    "last_ci_status": ws.get("last_ci_status", "none"),
    "git_dirty": str(ws.get("git_dirty", False)),
    "user_tier": meta.get("user_tier", "unknown"),
    "turn_index": meta.get("turn_index", 0),
    "budget_tokens_remaining_log": np.log1p(meta.get("budget_tokens_remaining", 0)),
    "loc_log": np.log1p(ws.get("loc", 0)),
    "n_open_files": len(ws.get("open_files") or []),
    "dominant_code_lang": dom_lang,
    "dominant_code_ratio": dom_ratio,
    "n_code_langs": len(lang_mix),
    "elapsed_session_sec_log": np.log1p(meta.get("elapsed_session_sec", 0)),
}

# train_ids / val_ids 순서 그대로 매핑
session_train_df = pd.DataFrame([extract_session_features(i) for i in train_ids])
session_val_df = pd.DataFrame([extract_session_features(i) for i in val_ids])

In [ ]:
# history 추가 연결
def extract_history_features(sample_id):
    s = samples_by_id[sample_id]
    history = s.get("history", []) or []

    # 최근 assistant_action 찾기 (뒤에서부터 탐색)
    last_action = None
    turns_since = 0
    for turn in reversed(history):
        if turn.get("role") == "assistant_action":
            last_action = turn
            break
        turns_since += 1   # user 턴을 지나칠 때마다 카운트

    if last_action is None:
        # 아직 action이 한 번도 없었던 경우 (대화 시작 단계)
        return {
            "last_action_name": "none",
            "last_action_result_status": "none",
            "n_actions_so_far": 0,
            "last_action_ext": "none",
            "turns_since_last_action": len(history),
        }

    # 결과 요약에서 성공/실패/에러 신호 추출
    result_summary = (last_action.get("result_summary") or "").lower()
    if re.search(r"fail|error|exception|traceback", result_summary):
        result_status = "fail"
    elif re.search(r"\bok\b|success|passed", result_summary):
        result_status = "ok"
    else:
        result_status = "unknown"

    # 파일 확장자 추출 (args.path가 있을 경우)
    path = (last_action.get("args") or {}).get("path", "")
    ext_match = re.search(r"\.(\w+)$", path)
    ext = ext_match.group(1) if ext_match else "none"

    # 전체 action 개수 카운트
    n_actions = sum(1 for turn in history if turn.get("role") == "assistant_action")

    return {
        "last_action_name": last_action.get("name", "none"),
        "last_action_result_status": result_status,
        "n_actions_so_far": n_actions,
        "last_action_ext": ext,
        "turns_since_last_action": turns_since,
    }

history_train_df = pd.DataFrame([extract_history_features(i) for i in train_ids])
history_val_df = pd.DataFrame([extract_history_features(i) for i in val_ids])

In [ ]:
# 원-핫 인코딩 (학습 데이터로 fit, 검증은 transform만)
CATEGORICAL_COLS = ["language_pref", "last_ci_status", "git_dirty", "user_tier", "dominant_code_lang",
                    "last_action_name", "last_action_result_status", "last_action_ext"]
NUMERIC_COLS = ["turn_index", "budget_tokens_remaining_log", "loc_log", "n_open_files", "dominant_code_ratio", "n_code_langs", "elapsed_session_sec_log",
                "n_actions_so_far", "turns_since_last_action"]

# session_train_df와 history_train_df 합치기
combined_train_df = pd.concat([session_train_df, history_train_df], axis=1)
combined_val_df = pd.concat([session_val_df, history_val_df], axis=1)

combined_encoder = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL_COLS),
    ("num", StandardScaler(), NUMERIC_COLS),   # 숫자는 변환 없이 그대로 통과
])

combined_train_enc = combined_encoder.fit_transform(combined_train_df)
combined_val_enc = combined_encoder.transform(combined_val_df)

##4. 모델 정의와 학습 (TF-IDF + LightGBM)

In [ ]:
#규칙 기반 피처 클래스
class RuleFeatureExtractor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self   # 학습할 게 없으니 그냥 자기 자신 반환

    def transform(self, X):
        return np.array([self._extract(text) for text in X])

    def _extract(self, text):
        return [
            1 if re.search(r'\b(read|열어|읽어)\b', text, re.I) else 0,
            1 if re.search(r'(grep|검색|찾아).*(코드|텍스트|패턴)', text, re.I) else 0,
            1 if re.search(r'(폴더|디렉토리|directory).*(뭐|목록|구조)', text, re.I) else 0,
            1 if re.search(r'\*\.\w+|글롭|glob|확장자', text, re.I) else 0,
            1 if re.search(r'(수정|고쳐|바꿔|edit)', text, re.I) else 0,
            1 if re.search(r'(새로|생성|만들어|write).*(파일|코드)', text, re.I) else 0,
            1 if re.search(r'(diff|patch|패치|변경사항)', text, re.I) else 0,
            1 if re.search(r'(셸|shell|bash|명령어|터미널)', text, re.I) else 0,
            1 if re.search(r'(테스트|test).*(실행|돌려|run)', text, re.I) else 0,
            1 if re.search(r'(린트|lint|타입체크|맞춤법)', text, re.I) else 0,
            1 if '?' in text else 0,
            1 if re.search(r'(계획|설계|plan)', text, re.I) else 0,
            1 if re.search(r'(웹|인터넷|구글|web)', text, re.I) else 0,
            len(text),
            len(re.findall(r'[a-zA-Z]', text)),
        ]

rule_extractor = RuleFeatureExtractor()
rule_features_train = rule_extractor.transform(X_train)
rule_features_val = rule_extractor.transform(X_val)

In [ ]:
#LightGBM의 성능평가지표를 Macro-F1 직접 만들기

def macro_f1_eval(y_true, y_pred_proba):
    y_pred = y_pred_proba.reshape(len(y_true), -1).argmax(axis=1)
    f1 = f1_score(y_true, y_pred, average="macro")
    return "macro_f1", f1, True   # True = "높을수록 좋음"

In [ ]:
#1. TF-IDF
tfidf_chi2 = Pipeline([
    #1-1) TF-IDF
    ("tfidf", TfidfVectorizer(ngram_range=(1,3), min_df=2, max_features=80_000,
                                sublinear_tf=True, lowercase=True)),
    #1-2) chi2 피처 선택
    ("chi2", SelectKBest(chi2, k=30_000)),
])

# 가지2: 규칙 기반 피처
rule = RuleFeatureExtractor()

# 두 갈래를 FeatureUnion으로 결합
vectorizer_pipe = FeatureUnion([
    ("tfidf_chi2", tfidf_chi2),
    ("rule_features", rule),
])

vectorizer = vectorizer_pipe
X_train_vec = vectorizer.fit_transform(X_train, y_train)
X_val_vec = vectorizer.transform(X_val)

#session meta, history 피쳐 결합
X_train_final = hstack([X_train_vec, combined_train_enc])
X_val_final = hstack([X_val_vec, combined_val_enc])

In [ ]:
#2. LightGBM
clf = lgb.LGBMClassifier(n_estimators=400, num_leaves=100, min_child_samples=10, learning_rate=0.05,
                            objective="multiclass", class_weight="balanced", force_row_wise=True, random_state=42)
best_clf = clf

best_clf.fit(
    X_train_final, y_train,
    eval_set=[(X_val_final, y_val)],
    eval_metric=macro_f1_eval,
    callbacks=[lgb.early_stopping(stopping_rounds=150)],
)

[LightGBM] [Info] Total Bins 213233
[LightGBM] [Info] Number of data points in the train set: 56000, number of used features: 10524
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don'

,boosting_type,'gbdt'
,num_leaves,100
,max_depth,-1
,learning_rate,0.05
,n_estimators,400
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,10


In [ ]:
tfidf_probs_train = best_clf.predict_proba(X_train_final)   # 학습셋 확률 (가중치 튜닝용)
tfidf_probs_val = best_clf.predict_proba(X_val_final)       # 검증셋 확률

rule_only_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
rule_only_clf.fit(rule_features_train, y_train)
rule_probs_train = rule_only_clf.predict_proba(rule_features_train)
rule_probs_val = rule_only_clf.predict_proba(rule_features_val)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
#클래스별 확신도 threshold - 윤서영 추가

class_thresholds_final = {
    # 클래스별로 정확도 90% 달성하는 threshold
    "edit_file": 0.65,
    "write_file": 0.4,
    "apply_patch": 0.85,
    "run_bash": 0.9,
    "lint_or_typecheck": 0.95,
    "respond_only": 0.4,

    # 신뢰 불가 클래스 — threshold를 "절대 못 넘게" 설정해서 무조건 Stage2로
    "read_file": 1.01,
    "grep_search": 1.01,
    "list_directory": 1.01,
    "glob_pattern": 1.01,
    "web_search": 1.01,
    "plan_task": 1.01,
    "run_tests": 1.01,
    "ask_user": 1.01,
}

In [ ]:
best_w = 1.0

def route_by_confidence(ids, tfidf_probs, rule_probs, classes, class_thresholds, w=best_w): #윤서영 수정 - 클래스별로 다른 threshold 적용
    """
    Stage1 앙상블 예측 확률을 보고, 확신도에 따라 라우팅하는 함수

    Parameters
    ----------
    ids : list
        각 샘플의 고유 id
    tfidf_probs : np.ndarray (샘플수, 14)
        TF-IDF+LightGBM 모델의 확률
    rule_probs : np.ndarray (샘플수, 14)
        규칙피처+LogReg 모델의 확률
    classes: probs의 각 컬럼이 어떤 클래스를 의미하는지 알려주는 배열
        (반드시 predict_proba를 만든 모델의 .classes_ 를 그대로 넘길 것) ### 임유미 추가
    class_threshold : float
        이 값 이상이면 Stage1에서 바로 확정 (클래스별로 다름)
    w : float
        TF-IDF 쪽 가중치

    Returns
    -------
    stage1_results : dict {id: action}
        확신도 높아서 Stage1에서 바로 결정된 것들
    stage2_input_ids : list
        확신도 낮아서 Stage2로 넘겨야 하는 id 목록
    stage2_probs : np.ndarray
        Stage2로 넘어가는 샘플들의 Stage1 확률 분포 (best_clf.predict_proba(X_vec) 결과)
    """
    ensemble_probs = w * tfidf_probs + (1 - w) * rule_probs
    max_conf = ensemble_probs.max(axis=1)      # 각 샘플의 "가장 높은 확률"
    pred_idx = ensemble_probs.argmax(axis=1)   # 그 확률에 해당하는 클래스 인덱스

    stage1_results = {}
    stage2_input_ids = []
    stage2_probs_list = []

    for sample_id, conf, idx, prob_row in zip(ids, max_conf, pred_idx, ensemble_probs):
      cls = classes[idx]
      th = class_thresholds.get(cls, 0.95)
      if conf >= th:
        stage1_results[sample_id] = cls
      else:
        stage2_input_ids.append(sample_id)
        stage2_probs_list.append(prob_row)

    stage2_probs = np.array(stage2_probs_list) if stage2_probs_list else np.empty((0, len(ALL_CLASSES)))

    return stage1_results, stage2_input_ids, stage2_probs

In [ ]:
#stage2 들어갈 데이터 받는 예시 코드
stage1_results, stage2_input_ids, stage2_probs = route_by_confidence(
    ids=val_ids,
    tfidf_probs=tfidf_probs_val,
    rule_probs=rule_probs_val,
    classes=best_clf.classes_,   ### 임유미 추가
    class_thresholds=class_thresholds_final,  # 윤서영 추가
)

print(f"Stage1에서 확정된 샘플 수: {len(stage1_results)}")
print(f"Stage2로 넘어가는 샘플 수: {len(stage2_input_ids)}")

Stage1에서 확정된 샘플 수: 2961
Stage2로 넘어가는 샘플 수: 11039


In [ ]:
# 확정 정확도 재확인
ensemble_probs_val = best_w * tfidf_probs_val + (1-best_w) * rule_probs_val

y_val_arr = np.array(val_ids)   # id 매칭용
confirmed_pred = np.array([stage1_results[sid] for sid in val_ids if sid in stage1_results])
confirmed_true = np.array([y_val[i] for i, sid in enumerate(val_ids) if sid in stage1_results])
print(f"확정 정확도: {(confirmed_pred == confirmed_true).mean():.4f}")

##5. Stage1 검증 (Macro-F1)

In [ ]:
ensemble_pred = np.array([best_clf.classes_[i] for i in ensemble_probs_val.argmax(axis=1)])
macro_f1 = f1_score(y_val, ensemble_pred, labels=ALL_CLASSES, average="macro", zero_division=0)
print(f"Validation Macro-F1: {macro_f1:.4f}")

Validation Macro-F1: 0.5820


# Stage2

##1. 라이브러리 정의

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

## 2. Stage2 데이터셋 구성

In [ ]:
def build_stage2_input_text(sample, history_turns=3):
    history = sample.get("history", []) or []
    hist_text = "\n".join(str(turn) for turn in history[-history_turns:])
    return f"{hist_text}\n{sample.get('current_prompt', '')}".strip()

def build_stage2_dataset(stage2_ids, stage2_probs, samples_by_id, labels, session_encoder, history_turns=3):
    """
    stage2_ids/stage2_probs: route_by_confidence가 넘긴 것 그대로 (stage2_probs는 앙상블 확률)
    session_encoder: Stage1에서 이미 fit된 ColumnTransformer (train 기준) — transform만 함
    """
    texts, y_stage2 = [], []
    for sid in stage2_ids:
        s = samples_by_id[sid]
        texts.append(build_stage2_input_text(s, history_turns))
        y_stage2.append(labels[sid])

    # ---- session_meta 인코딩 (Stage1의 extract_session_features 재사용) ----
    session_df = pd.DataFrame([extract_session_features(sid) for sid in stage2_ids])
    history_df = pd.DataFrame([extract_history_features(sid) for sid in stage2_ids])   ### 윤서영 추가
    combined_df = pd.concat(   ### 윤서영 추가
        [session_df.reset_index(drop=True), history_df.reset_index(drop=True)], axis=1
    )
    session_enc = session_encoder.transform(combined_df)   ### 윤서영 수정
    if hasattr(session_enc, "toarray"):   # ColumnTransformer 결과가 sparse일 수 있음
        session_enc = session_enc.toarray()

    return texts, y_stage2, np.array(stage2_probs), session_enc.astype(np.float32)

X_stage2_text, y_stage2, stage1_probs_stage2, session_stage2_enc = build_stage2_dataset(
    stage2_input_ids, stage2_probs, samples_by_id, labels, combined_encoder,
)
print(f"Stage2 학습 후보: {len(X_stage2_text)}건")
print(f"session_meta 인코딩 shape: {session_stage2_enc.shape}")
print(pd.Series(y_stage2).value_counts())

print("session_stage2_enc shape:", session_stage2_enc.shape)  # (샘플수, 26)이어야 함
print(combined_encoder.get_feature_names_out())  # 'num__elapsed_session_sec_log' 포함 확인

Stage2 학습 후보: 11039건
session_meta 인코딩 shape: (11039, 75)
grep_search          1980
read_file            1849
edit_file            1432
glob_pattern         1056
list_directory        866
run_tests             802
run_bash              779
apply_patch           649
ask_user              485
plan_task             471
lint_or_typecheck     405
web_search            240
write_file             20
respond_only            5
Name: count, dtype: int64
session_stage2_enc shape: (11039, 75)
['cat__language_pref_en' 'cat__language_pref_ko'
 'cat__language_pref_mixed' 'cat__last_ci_status_failed'
 'cat__last_ci_status_none' 'cat__last_ci_status_passed'
 'cat__git_dirty_False' 'cat__git_dirty_True' 'cat__user_tier_enterprise'
 'cat__user_tier_free' 'cat__user_tier_pro' 'cat__dominant_code_lang_go'
 'cat__dominant_code_lang_java' 'cat__dominant_code_lang_py'
 'cat__dominant_code_lang_rs' 'cat__dominant_code_lang_ts'
 'cat__dominant_code_lang_tsx' 'cat__dominant_code_lang_vue'
 'cat__dominant_code_lan

##3. 분류 헤드 (K-fold로 stage2_input_ids 전체 예측 만들기)

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_stage2_arr = np.array(y_stage2)
oof_preds = np.empty(len(y_stage2_arr), dtype=object)  # out-of-fold 예측 저장용

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
emb_all = embedder.encode(X_stage2_text, show_progress_bar=True)  # 전체 텍스트 한 번에 임베딩

feat_all = np.hstack([emb_all, stage1_probs_stage2, session_stage2_enc])

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(feat_all, y_stage2_arr)):
    head_fold = LogisticRegression(max_iter=1000, class_weight="balanced")
    head_fold.fit(feat_all[train_idx], y_stage2_arr[train_idx])
    oof_preds[val_idx] = head_fold.predict(feat_all[val_idx])
    print(f"Fold {fold_idx+1}/{n_splits} 완료")

# 이제 stage2_input_ids 전체에 대해 예측이 다 채워짐
stage2_oof_dict = dict(zip(stage2_input_ids, oof_preds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/345 [00:00<?, ?it/s]

Fold 1/5 완료
Fold 2/5 완료
Fold 3/5 완료
Fold 4/5 완료
Fold 5/5 완료


In [ ]:
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(feat_all, y_stage2_arr)):
    head_fold = LogisticRegression(max_iter=1000, class_weight="balanced")
    head_fold.fit(feat_all[train_idx], y_stage2_arr[train_idx])
    oof_preds[val_idx] = head_fold.predict(feat_all[val_idx])
    print(f"Fold {fold_idx+1}/{n_splits} 완료")

stage2_oof_dict = dict(zip(stage2_input_ids, oof_preds))

# ---- 전체 stage2 데이터로 최종 head 학습 ----
head_final = LogisticRegression(max_iter=1000, class_weight="balanced")
head_final.fit(feat_all, y_stage2_arr)

Fold 1/5 완료
Fold 2/5 완료
Fold 3/5 완료
Fold 4/5 완료
Fold 5/5 완료


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

##4. Stage2 Macro-F1 측정

In [ ]:
"""##4. Stage2 Macro-F1 측정"""

stage2_macro_f1 = f1_score(y_stage2_arr, oof_preds, labels=ALL_CLASSES, average="macro", zero_division=0)
print(f"Stage2 단독 Validation Macro-F1 (OOF): {stage2_macro_f1:.4f}")
print(classification_report(y_stage2_arr, oof_preds, labels=ALL_CLASSES, zero_division=0))

Stage2 단독 Validation Macro-F1 (OOF): 0.4872
                   precision    recall  f1-score   support

        read_file       0.50      0.38      0.43      1849
      grep_search       0.61      0.50      0.55      1980
   list_directory       0.34      0.58      0.43       866
     glob_pattern       0.51      0.55      0.53      1056
        edit_file       0.80      0.70      0.75      1432
       write_file       0.16      0.25      0.20        20
      apply_patch       0.52      0.69      0.60       649
         run_bash       0.67      0.60      0.63       779
        run_tests       0.63      0.64      0.64       802
lint_or_typecheck       0.45      0.59      0.51       405
         ask_user       0.51      0.49      0.50       485
        plan_task       0.52      0.46      0.49       471
       web_search       0.42      0.60      0.49       240
     respond_only       0.06      0.20      0.09         5

         accuracy                           0.55     11039
        ma

#Stage1+Stage2 전체 Macro-F1 측정

In [ ]:
##5. Stage1 + Stage2 통합 Macro-F1 (전체 val_ids 커버)

final_preds = dict(stage1_results)          # stage1 확정분 1,643건
final_preds.update(stage2_oof_dict)         # stage2 전체 12,357건 (OOF 예측)

final_ids = list(final_preds.keys())
print(f"통합 샘플 수: {len(final_ids)}")    # 이제 14,000이 나와야 정상

y_true_final = [labels[i] for i in final_ids]
y_pred_final = [final_preds[i] for i in final_ids]

overall_macro_f1 = f1_score(y_true_final, y_pred_final, labels=ALL_CLASSES, average="macro", zero_division=0)
print(f"🎯 Stage1+Stage2 통합 Validation Macro-F1: {overall_macro_f1:.4f}")
print(classification_report(y_true_final, y_pred_final, labels=ALL_CLASSES, zero_division=0))

통합 샘플 수: 14000
🎯 Stage1+Stage2 통합 Validation Macro-F1: 0.6300
                   precision    recall  f1-score   support

        read_file       0.50      0.38      0.43      1851
      grep_search       0.61      0.50      0.55      1982
   list_directory       0.34      0.58      0.43       866
     glob_pattern       0.51      0.55      0.53      1057
        edit_file       0.87      0.80      0.83      2234
       write_file       0.91      0.94      0.93       296
      apply_patch       0.64      0.77      0.70       965
         run_bash       0.73      0.68      0.71      1014
        run_tests       0.66      0.66      0.66       912
lint_or_typecheck       0.48      0.60      0.53       457
         ask_user       0.54      0.50      0.52       540
        plan_task       0.55      0.51      0.53       536
       web_search       0.43      0.60      0.50       255
     respond_only       0.98      1.00      0.99      1035

         accuracy                           0.63   

#전체 데이터로 재학습 & 모델 저장

In [ ]:
# 전체 학습 데이터로 재학습
vectorizer_final = vectorizer_pipe
X_full_vec = vectorizer_final.fit_transform(X, y)

# session_meta, history도 전체 데이터 기준으로 재학습 ---
session_full_df = pd.DataFrame([extract_session_features(i) for i in ids])
history_full_df = pd.DataFrame([extract_history_features(i) for i in ids])


# session + history 결합
combined_full_df = pd.concat(
    [session_full_df.reset_index(drop=True), history_full_df.reset_index(drop=True)],
    axis=1,
)

session_encoder_final = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL_COLS),
    ("num", StandardScaler(), NUMERIC_COLS),
])
session_full_enc = session_encoder_final.fit_transform(combined_full_df)

X_full_final = hstack([X_full_vec, session_full_enc])   # session_meta 포함!

clf_final = clf
clf_final.fit(X_full_final, y)

rule_features_full = rule_extractor.transform(X)
rule_only_clf_final = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
rule_only_clf_final.fit(rule_features_full, y)

[LightGBM] [Info] Total Bins 266746
[LightGBM] [Info] Number of data points in the train set: 70000, number of used features: 13263
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not w

#stage1 경량화 - 윤서영 추가

In [ ]:
from onnxmltools.convert import convert_lightgbm
from onnxmltools.convert.common.data_types import FloatTensorType

#LightGBM -> ONNX 변환
n_features = X_full_final.shape[1]   # 학습에 쓴 전체 피처 개수
initial_type = [('input', FloatTensorType([None, n_features]))]

onnx_model = convert_lightgbm(
    clf_final,
    initial_types=initial_type,
    target_opset=13,
    zipmap=False,   #확률 출력이 딕셔너리(ZipMap)가 아니라 배열로 나오게
)

In [ ]:
# 저장 (추론용 script.py가 ./model/tfidf_lgbm.onnx 을 불러옵니다)
os.makedirs("./model", exist_ok=True)

joblib.dump(vectorizer_final, "./model/vectorizer.pkl", compress=3)
joblib.dump(session_encoder_final, "./model/session_encoder.pkl", compress=3)  # 임유미  추가
with open("./model/tfidf_lgbm.onnx", "wb") as f: #윤서영 수정
    f.write(onnx_model.SerializeToString())
joblib.dump(rule_only_clf_final, "./model/rule_logreg.pkl", compress=3)
joblib.dump(head_final, "./model/stage2_head.pkl", compress=3) # 임유미 추가

print("저장 완료: ./model/tfidf_lgbm.onnx")

저장 완료: ./model/tfidf_lgbm.onnx


# 추론

# 1. 테스트 데이터 불러오기

In [ ]:
TEST_PATH = os.path.join(DATA_DIR, "test.jsonl")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
OUT_DIR = "/content/drive/MyDrive/nlp_competition/output"   ### 경로 필요에 따라 수정
OUT_PATH = os.path.join(OUT_DIR, "submission.csv")

test_samples = [json.loads(line)
                for line in open(TEST_PATH, encoding="utf-8")
                if line.strip()]

test_ids = [s["id"] for s in test_samples]
test_texts = [s.get("current_prompt", "") or "" for s in test_samples]
test_samples_by_id = {s["id"]: s for s in test_samples}

print("test samples:", len(test_samples))

test samples: 5


# 2. 테스트용 session_meta, history 함수

In [ ]:
def extract_session_features_infer(sample):
    meta = sample.get("session_meta", {}) or {}
    ws = meta.get("workspace", {}) or {}
    lang_mix = ws.get("language_mix") or {}
    dom_lang, dom_ratio = max(lang_mix.items(), key=lambda kv: kv[1]) if lang_mix else ("none", 0.0)

    return {
        "language_pref": meta.get("language_pref", "unknown"),
        "last_ci_status": ws.get("last_ci_status", "none"),
        "git_dirty": str(ws.get("git_dirty", False)),
        "user_tier": meta.get("user_tier", "unknown"),
        "turn_index": meta.get("turn_index", 0),
        "budget_tokens_remaining_log": np.log1p(meta.get("budget_tokens_remaining", 0)),
        "loc_log": np.log1p(ws.get("loc", 0)),
        "n_open_files": len(ws.get("open_files") or []),
        "dominant_code_lang": dom_lang,
        "dominant_code_ratio": dom_ratio,
        "n_code_langs": len(lang_mix),
        "elapsed_session_sec_log": np.log1p(meta.get("elapsed_session_sec", 0)),
}

In [ ]:
def extract_history_features_infer(sample_id): #윤서영 수정
    history = sample.get("history", []) or []

    last_action = None
    turns_since = 0
    for turn in reversed(history):
        if turn.get("role") == "assistant_action":
            last_action = turn
            break
        turns_since += 1

    if last_action is None:
        return {
            "last_action_name": "none",
            "last_action_result_status": "none",
            "n_actions_so_far": 0,
            "last_action_ext": "none",
            "turns_since_last_action": len(history),
        }

    result_summary = (last_action.get("result_summary") or "").lower()
    if re.search(r"fail|error|exception|traceback", result_summary):
        result_status = "fail"
    elif re.search(r"\bok\b|success|passed", result_summary):
        result_status = "ok"
    else:
        result_status = "unknown"

    path = (last_action.get("args") or {}).get("path", "")
    ext_match = re.search(r"\.(\w+)$", path)
    ext = ext_match.group(1) if ext_match else "none"

    n_actions = sum(1 for turn in history if turn.get("role") == "assistant_action")

    return {
        "last_action_name": last_action.get("name", "none"),
        "last_action_result_status": result_status,
        "n_actions_so_far": n_actions,
        "last_action_ext": ext,
        "turns_since_last_action": turns_since,
    }

# 3. Stage1 특징 생성 + 확률 계산

In [ ]:
test_X_vec = vectorizer_final.transform(test_texts)
test_rule_features = rule_extractor.transform(test_texts)

test_session_df = pd.DataFrame([extract_session_features_infer(s) for s in test_samples])
test_history_df = pd.DataFrame([extract_history_features_infer(s) for s in test_samples]) #윤서영 추가
test_combined_df = pd.concat( #윤서영 추가
    [test_session_df.reset_index(drop=True), test_history_df.reset_index(drop=True)], axis=1
)

test_session_enc = session_encoder_final.transform(test_combined_df) #윤서영 수정

test_X_final = hstack([test_X_vec, test_session_enc])

test_tfidf_probs = clf_final.predict_proba(test_X_final)
test_rule_probs = rule_only_clf_final.predict_proba(test_rule_features)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


# 4. Stage1 라우팅

In [ ]:
test_stage1_results, test_stage2_input_ids, test_stage2_probs = route_by_confidence(
    ids=test_ids,
    tfidf_probs=test_tfidf_probs,
    rule_probs=test_rule_probs,
    classes=clf_final.classes_,
    class_thresholds=class_thresholds_final,  # 윤서영 추가
    w=best_w,
)
print(f"Stage1 확정: {len(test_stage1_results)} | Stage2로: {len(test_stage2_input_ids)}")

Stage1 확정: 2 | Stage2로: 3


# 5. Stage2 처리

In [ ]:
test_stage2_results = {}
if test_stage2_input_ids:
    test_stage2_texts = [
        build_stage2_input_text(test_samples_by_id[sid]) for sid in test_stage2_input_ids
    ]
    test_stage2_session_df = pd.DataFrame(
        [extract_session_features_infer(test_samples_by_id[sid]) for sid in test_stage2_input_ids]
    )
    test_stage2_history_df = pd.DataFrame( #윤서영 추가
        [extract_history_features_infer(test_samples_by_id[sid]) for sid in test_stage2_input_ids]
    )
    test_stage2_combined_df = pd.concat( #윤서영 추가
        [test_stage2_session_df.reset_index(drop=True), test_stage2_history_df.reset_index(drop=True)], axis=1
    )

    test_stage2_session_enc = session_encoder_final.transform(test_stage2_combined_df) #윤서영 수정
    if hasattr(test_stage2_session_enc, "toarray"):
        test_stage2_session_enc = test_stage2_session_enc.toarray()

    # embedder는 #19에서 이미 로드해둔 것을 그대로 재사용 (콜랩은 인터넷이 되므로
    # 이름으로 불러온 것 그대로 써도 무방. 오프라인 평가서버 제출용 별도
    # script.py에서는 로컬 경로(minilm_local)로 바꿔야 함 — 이전 안내 참고)
    test_stage2_emb = embedder.encode(test_stage2_texts, show_progress_bar=True)

    test_stage2_feat = np.hstack([test_stage2_emb, test_stage2_probs, test_stage2_session_enc])
    test_stage2_preds = head_final.predict(test_stage2_feat)
    test_stage2_results = dict(zip(test_stage2_input_ids, test_stage2_preds))
    print(f"Stage2 예측 완료: {len(test_stage2_results)}건")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Stage2 예측 완료: 3건


# 6. 결과 합치기 + submission.csv 저장

In [ ]:
final_test_preds = {**test_stage1_results, **test_stage2_results}
preds_in_order = [str(final_test_preds.get(i, "")) for i in test_ids]

with open(SAMPLE_SUB_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    sub_rows = list(reader)

pred_map = dict(zip(test_ids, preds_in_order))
n_missing = 0
for row in sub_rows:
    p = pred_map.get(row["id"])
    if p is None:
        n_missing += 1
    else:
        row["action"] = p
if n_missing:
    print(f" 경고: 예측이 없어 placeholder를 유지한 id {n_missing}건")

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sub_rows)

print(f"Saved: {OUT_PATH} (rows={len(sub_rows)})")

Saved: /content/drive/MyDrive/nlp_competition/output/submission.csv (rows=5)


In [ ]:
!pip freeze | grep -iE "scikit-learn|lightgbm|joblib|sentence-transformers|scipy|numpy|pandas"

geopandas==1.1.3
joblib==1.5.3
lightgbm==4.6.0
numpy==2.0.2
pandas==2.2.2
pandas-datareader==0.11.1
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
scikit-learn==1.6.1
scipy==1.16.3
sentence-transformers==5.6.0
sklearn-pandas==2.2.0


In [ ]:
import transformers, tokenizers, sentence_transformers
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("sentence_transformers:", sentence_transformers.__version__)

transformers: 5.12.1
tokenizers: 0.22.2
sentence_transformers: 5.6.0


In [ ]:
import torch, huggingface_hub, safetensors
print("torch:", torch.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("safetensors:", safetensors.__version__)

torch: 2.11.0+cu128
huggingface_hub: 1.20.1
safetensors: 0.8.0


In [ ]:
# ----- 신규 셀: model/ 폴더를 nlp_competition_3로 복사 -----
# 셀 39는 그대로 유지 (팀원 참조용). 이 셀은 셀 39 실행 이후,
# 그 결과물(./model)을 별도 드라이브 폴더로 복사만 하는 용도.

import shutil

SOURCE_DIR = "./model"
DEST_DIR = "/content/drive/MyDrive/nlp_competition_3/model"

os.makedirs(DEST_DIR, exist_ok=True)
shutil.copytree(SOURCE_DIR, DEST_DIR, dirs_exist_ok=True)

print("복사 완료:", DEST_DIR)
print("복사된 파일:", os.listdir(DEST_DIR))

복사 완료: /content/drive/MyDrive/nlp_competition_3/model
복사된 파일: ['rule_logreg.pkl', 'session_encoder.pkl', 'vectorizer.pkl', 'tfidf_lgbm.onnx', 'stage2_head.pkl']


In [ ]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(onnx_model.SerializeToString())
input_name = sess.get_inputs()[0].name
print("입력 이름:", input_name)

X_sample_dense = X_val_final[:5].toarray().astype(np.float32)
outputs = sess.run(None, {input_name: X_sample_dense})

print("출력 개수:", len(outputs))
for i, o in enumerate(outputs):
    arr = np.array(o)
    print(f"outputs[{i}] shape: {arr.shape}, dtype: {arr.dtype}")
print("outputs[0] 예시:", outputs[0][:3])
print("outputs[1] 예시:", outputs[1][:3] if len(outputs) > 1 else "없음")

입력 이름: input
출력 개수: 2
outputs[0] shape: (5,), dtype: object
outputs[1] shape: (5, 14), dtype: float32
outputs[0] 예시: ['apply_patch' 'run_tests' 'run_bash']
outputs[1] 예시: [[3.21619004e-01 7.05963895e-02 1.52271926e-01 1.06289528e-01
  1.44617006e-01 5.52689657e-04 2.02301107e-02 1.30834831e-02
  1.10256121e-01 2.96766331e-07 5.95732331e-02 8.45370349e-04
  6.47257766e-05 1.83932727e-08]
 [2.49762591e-02 1.49461941e-03 7.52312038e-03 6.82910439e-03
  5.14853839e-03 2.10883489e-04 3.20323789e-03 3.63320869e-04
  5.80461416e-03 3.85133006e-07 6.87102824e-02 8.75726163e-01
  9.52750270e-06 6.77863454e-09]
 [1.82504043e-01 6.16603810e-03 4.82726134e-02 1.21766806e-01
  1.40707538e-01 6.13976852e-04 6.97904825e-02 1.12404181e-02
  9.65968445e-02 2.58370551e-06 3.21722686e-01 5.68935589e-04
  4.70541199e-05 2.44585312e-08]]


In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive/nlp_competition_3/model/minilm_local"))

False
